# 10 — Practice: Module 2 — IO & Memory

20 interview-style questions on file reading/writing, large file handling, memory optimization,
and Copy-on-Write. Every question has a complete, working solution.

In [ ]:
import pandas as pd
import numpy as np
import tempfile
import os
from io import StringIO

TMPDIR = tempfile.mkdtemp()

np.random.seed(42)
n = 10_000

transactions = pd.DataFrame({
    'txn_id':   range(1, n + 1),
    'customer': np.random.randint(1, 1000, n),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], n),
    'amount':   np.round(np.random.uniform(50, 5000, n), 2),
    'city':     np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Chennai'], n),
    'returned': np.random.choice([0, 1], n, p=[0.92, 0.08]),
    'date':     pd.date_range('2023-01-01', periods=n, freq='h').strftime('%Y-%m-%d')
})

csv_path = os.path.join(TMPDIR, 'transactions.csv')
transactions.to_csv(csv_path, index=False)

print(f"Dataset: {transactions.shape}, CSV: {os.path.getsize(csv_path)/1024:.1f} KB")

---
### Q1: Read a CSV using only specific columns and specific dtypes

In [ ]:
df = pd.read_csv(
    csv_path,
    usecols=['txn_id', 'category', 'amount', 'returned'],
    dtype={'txn_id': 'int32', 'amount': 'float32', 'returned': 'int8', 'category': 'category'}
)
print(df.dtypes)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

---
### Q2: Handle a messy CSV with custom separator, NA values, and skip rows

In [ ]:
messy = """# Report generated 2024-01-01
ID;Category;Amount;Status
1;Electronics;5,999;OK
2;Books;N/A;OK
3;Clothing;1,200;NULL
4;Food;350;OK
"""

df_messy = pd.read_csv(
    StringIO(messy),
    sep=';',
    skiprows=1,
    na_values=['N/A', 'NULL'],
    thousands=','
)
print(df_messy)
print(df_messy.dtypes)

---
### Q3: Write to multiple Excel sheets

In [ ]:
excel_path = os.path.join(TMPDIR, 'report.xlsx')

try:
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for cat in transactions['category'].unique():
            subset = transactions[transactions['category'] == cat]
            subset.to_excel(writer, sheet_name=cat, index=False)

    print(f"Excel written: {os.path.getsize(excel_path)/1024:.0f} KB")

    all_sheets = pd.read_excel(excel_path, sheet_name=None)
    for name, df in all_sheets.items():
        print(f"  {name}: {df.shape}")
except Exception as e:
    print(f"openpyxl needed: pip install openpyxl | Error: {e}")

---
### Q4: Write and read Parquet, compare size with CSV

In [ ]:
pq_path = os.path.join(TMPDIR, 'transactions.parquet')

try:
    transactions.to_parquet(pq_path, index=False, compression='snappy')

    csv_size = os.path.getsize(csv_path) / 1024
    pq_size  = os.path.getsize(pq_path) / 1024

    print(f"CSV:     {csv_size:.1f} KB")
    print(f"Parquet: {pq_size:.1f} KB")
    print(f"Saved:   {(1 - pq_size/csv_size):.1%}")

    df_pq = pd.read_parquet(pq_path)
    print(f"Read back: {df_pq.shape}")
except ImportError:
    print("pip install pyarrow")

---
### Q5: Use chunked reading to compute total revenue by category

In [ ]:
revenue_by_cat = {}

for chunk in pd.read_csv(csv_path, chunksize=2000):
    valid = chunk[chunk['returned'] == 0]
    agg = valid.groupby('category')['amount'].sum()
    for cat, total in agg.items():
        revenue_by_cat[cat] = revenue_by_cat.get(cat, 0) + total

result = pd.Series(revenue_by_cat, name='revenue').sort_values(ascending=False)
print(result.map(lambda x: f"{x:,.0f}"))

---
### Q6: Convert a naive read to an optimized read — show memory reduction

In [ ]:
df_naive = pd.read_csv(csv_path)
mem_naive = df_naive.memory_usage(deep=True).sum() / 1024

df_opt = pd.read_csv(
    csv_path,
    dtype={
        'txn_id':   'int32',
        'customer': 'int32',
        'category': 'category',
        'amount':   'float32',
        'city':     'category',
        'returned': 'int8',
    }
)
mem_opt = df_opt.memory_usage(deep=True).sum() / 1024

print(f"Naive:     {mem_naive:.1f} KB")
print(f"Optimized: {mem_opt:.1f} KB")
print(f"Reduction: {(1 - mem_opt/mem_naive):.1%}")

---
### Q7: Write a generic optimize_memory() function

In [ ]:
def optimize_memory(df: pd.DataFrame, cat_threshold: float = 0.5) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            df[col] = pd.to_numeric(df[col], downcast='integer')
        elif pd.api.types.is_float_dtype(dtype):
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif dtype == object and df[col].nunique() / len(df) < cat_threshold:
            df[col] = df[col].astype('category')
    return df

df_auto = optimize_memory(df_naive)
mem_auto = df_auto.memory_usage(deep=True).sum() / 1024
print(f"Auto-optimized: {mem_auto:.1f} KB (from {mem_naive:.1f} KB)")
print(df_auto.dtypes)

---
### Q8: Read from a URL (Titanic) with fallback

In [ ]:
try:
    titanic = pd.read_csv(
        'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    )
    print(f"Titanic: {titanic.shape}")
    print(titanic[['PassengerId', 'Survived', 'Pclass', 'Name']].head(3))
except Exception:
    print("URL unavailable — in production use: pd.read_csv(url, storage_options={...})")

---
### Q9: SQL round-trip — write to SQLite, query back with aggregate SQL

In [ ]:
import sqlite3

conn = sqlite3.connect(os.path.join(TMPDIR, 'sales.db'))
transactions.to_sql('txns', conn, if_exists='replace', index=False)

result = pd.read_sql("""
    SELECT city,
           COUNT(*) as orders,
           ROUND(SUM(amount), 2) as revenue,
           ROUND(AVG(amount), 2) as avg_order
    FROM txns
    WHERE returned = 0
    GROUP BY city
    ORDER BY revenue DESC
""", conn)

print(result)
conn.close()

---
### Q10: Demonstrate SettingWithCopyWarning — cause and fix

In [ ]:
import warnings

pd.options.mode.copy_on_write = False  # Classic mode to show the warning

df = transactions.head(20).copy()

# CAUSE: chained write
print("CAUSE: chained indexing for write")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    df[df['returned'] == 1]['amount'] = 0  # may or may not work
    if w:
        print(f"  Warning triggered: {w[0].category.__name__}")

# FIX: single loc
df.loc[df['returned'] == 1, 'amount'] = 0
print("FIX: df.loc[mask, col] = val — always correct")
print(df[df['returned'] == 1][['txn_id', 'amount']].head())

---
### Q11: Enable CoW and show that subset modification doesn't affect original

In [ ]:
pd.options.mode.copy_on_write = True

df = pd.DataFrame({'A': [1, 2, 3], 'B': [10, 20, 30]})
original_a0 = df.loc[0, 'A']

subset = df.iloc[0:2]
subset['A'] = 999  # with CoW: only modifies subset

print(f"Original A[0] unchanged: {df.loc[0, 'A'] == original_a0}")
print(f"subset A[0] = {subset.loc[0, 'A']}")

---
### Q12: Prove fancy indexing always returns a copy

In [ ]:
pd.options.mode.copy_on_write = False

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
fancy = df[['A', 'B']]  # fancy indexing

fancy.loc[0, 'A'] = 999

print(f"Original A[0]: {df.loc[0, 'A']} (unchanged)")
print(f"Fancy    A[0]: {fancy.loc[0, 'A']}")
print(f"shares_memory: {np.shares_memory(df['A'].values, fancy['A'].values)}")

---
### Q13: Read JSON orient='records' and 'split' — show difference

In [ ]:
sample = transactions.head(3)[['txn_id', 'category', 'amount']]

json_records = sample.to_json(orient='records')
json_split   = sample.to_json(orient='split')

print("records orient:")
print(json_records)

print("\nsplit orient:")
print(json_split)

# Read back
df_r = pd.read_json(StringIO(json_records), orient='records')
df_s = pd.read_json(StringIO(json_split), orient='split')
print("\nBoth read back correctly:", df_r.equals(df_s))

---
### Q14: Use nrows to sample and infer dtypes for a large file

In [ ]:
# Common pattern: read 1000 rows to understand structure before processing all rows
sample = pd.read_csv(csv_path, nrows=1000)

print("Schema from sample:")
print(sample.dtypes)
print(f"\nColumn cardinality:")
for col in sample.columns:
    n_unique = sample[col].nunique()
    print(f"  {col:<12} {n_unique} unique values")

---
### Q15: Chunked ETL — collect only returned transactions

In [ ]:
returned_chunks = []

for chunk in pd.read_csv(csv_path, chunksize=2500,
                         dtype={'returned': 'int8', 'amount': 'float32', 'category': 'category'}):
    returned_chunks.append(chunk[chunk['returned'] == 1][['txn_id', 'category', 'amount', 'city']])

returned_df = pd.concat(returned_chunks, ignore_index=True)
print(f"Total returned: {len(returned_df)}")
print(returned_df['category'].value_counts())

---
### Q16: Convert object columns to category and show per-column savings

In [ ]:
df = pd.read_csv(csv_path)
obj_cols = df.select_dtypes(include='object').columns.tolist()

print("Memory savings from object → category:")
for col in obj_cols:
    before = df[col].memory_usage(deep=True)
    after  = df[col].astype('category').memory_usage(deep=True)
    print(f"  {col:<12} {before:>8,} B → {after:>6,} B  ({(1 - after/before):.1%} saved)")

---
### Q17: Write a DataFrame to an in-memory buffer (BytesIO) and read it back

In [ ]:
from io import BytesIO

buf = BytesIO()
transactions.head(100).to_parquet(buf, index=False)
buf.seek(0)  # rewind to start

try:
    df_from_buf = pd.read_parquet(buf)
    print(f"From BytesIO: {df_from_buf.shape}")
    print(df_from_buf.head(3))
except Exception:
    buf = BytesIO()
    transactions.head(100).to_csv(buf, index=False)
    buf.seek(0)
    df_from_buf = pd.read_csv(buf)
    print(f"CSV from BytesIO: {df_from_buf.shape}")

---
### Q18: Demonstrate that modifying a view modifies the original (pre-CoW)

In [ ]:
pd.options.mode.copy_on_write = False

df = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': [10, 20, 30, 40, 50]})
col_a = df['A']  # may be a view of column A

original_val = df.loc[0, 'A']

# In classic Pandas, modifying via the Series may reflect in df
print(f"Shares memory: {np.shares_memory(df['A'].values, col_a.values)}")
print(f"Is view: {col_a._is_view}")
print(f"Original A[0]: {df.loc[0, 'A']}")

---
### Q19: Use skiprows with a callable to skip specific row patterns

In [ ]:
# skiprows with callable: skip every 3rd data row (downsample)
# Row 0 = header (always kept), rows 1..N are data
df_every_third = pd.read_csv(
    csv_path,
    skiprows=lambda i: i > 0 and i % 3 != 0  # keep row 0 (header) + every 3rd row
)
print(f"Original rows: {len(transactions):,}")
print(f"Every 3rd row: {len(df_every_third):,}")

---
### Q20: Full memory optimization pipeline — before/after comparison

In [ ]:
# Step 1: Naive read
df_naive = pd.read_csv(csv_path)
mem_naive = df_naive.memory_usage(deep=True).sum() / 1024

# Step 2: Smart read with dtypes
df_smart = pd.read_csv(
    csv_path,
    usecols=['txn_id', 'customer', 'category', 'amount', 'city', 'returned'],
    dtype={
        'txn_id':   'int32',
        'customer': 'int32',
        'category': 'category',
        'amount':   'float32',
        'city':     'category',
        'returned': 'int8',
    }
)
mem_smart = df_smart.memory_usage(deep=True).sum() / 1024

print(f"Naive read:    {mem_naive:.1f} KB")
print(f"Smart read:    {mem_smart:.1f} KB")
print(f"Reduction:     {(1 - mem_smart/mem_naive):.1%}")
print()
print(df_smart.dtypes)

---
## Module 2 Summary

| Topic | Key Insight |
|-------|-------------|
| CSV params | `usecols`, `dtype`, `parse_dates`, `na_values`, `thousands` |
| Format choice | Parquet for analytics, CSV for exchange, SQL for persistence |
| Large files | `chunksize` for >RAM files, `nrows` for exploration |
| Memory | `category` dtype saves 70-90% on low-cardinality strings |
| Copy vs View | Fancy indexing → copy; `df['col']` → may be view |
| CoW | Pandas 2.0: `pd.options.mode.copy_on_write = True` |
| Safe write | Always `df.loc[mask, col] = val` — never chained indexing |